In [ ]:
import duckdb
import pandas as pd

# Simulate Role-Based Access Control and Column Masking locally
con = duckdb.connect(':memory:')

data = [
    (1, "123-45-6789", "MRN-001", "Asthma"),
    (2, "987-65-4321", "MRN-002", "Diabetes")
]
df = pd.DataFrame(data, columns=["person_id", "ssn", "mrn", "condition"])
con.execute("CREATE TABLE silver_patients AS SELECT * FROM df")

def query_as_role(role_name):
    print(f"\n--- Querying as Role: {role_name} ---")
    if role_name == 'CLINICAL_ANALYST':
        # Has access to cleartext
        query = "SELECT person_id, ssn, mrn, condition FROM silver_patients"
    else:
        # Data Scientist - masked PII
        query = """
        SELECT 
            person_id, 
            '***-**-****' as ssn, 
            'REDACTED_MRN' as mrn, 
            condition 
        FROM silver_patients
        """
    print(con.execute(query).df())

query_as_role('DATA_SCIENTIST')
query_as_role('CLINICAL_ANALYST')

# Simulate dbt Gold Mart logic (30-day readmission)
admissions_data = [
    (1, '2023-01-01'),
    (1, '2023-01-15'), # 14 days (readmission!)
    (2, '2023-02-01'),
    (2, '2023-04-01')  # 60 days (not a readmission)
]
admissions_df = pd.DataFrame(admissions_data, columns=["patient_id", "admission_date"])
con.execute("CREATE TABLE condition_occurrence AS SELECT patient_id, CAST(admission_date AS DATE) as admission_date FROM admissions_df")

gold_mart_sql = """
WITH readmissions AS (
    SELECT 
        patient_id,
        admission_date,
        LEAD(admission_date) OVER (PARTITION BY patient_id ORDER BY admission_date) as next_admission_date
    FROM condition_occurrence
)
SELECT 
    patient_id,
    COUNT(*) as total_admissions,
    SUM(CASE WHEN next_admission_date - admission_date <= 30 THEN 1 ELSE 0 END) as thirty_day_readmissions
FROM readmissions
GROUP BY patient_id
"""

print("\n--- dbt Gold Mart: 30-Day Readmissions ---")
print(con.execute(gold_mart_sql).df())
